# 02 - Feature engineering and leakage checks

Every feature at row `t` uses data up to the close of day `t`. `target` and `next_ret` are labels only.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src import data as dmod
from src.features import build_feature_list, feature_groups, make_features, split_time_series
from src.utils import FIGURES_DIR, ensure_dirs

ensure_dirs()
raw = dmod.load_dataset('AAPL')
df = make_features(raw)
features = build_feature_list(df)
print(f'{len(df)} rows, {len(features)} features')
df[features].head()

## Leakage sanity checks

In [ ]:
# target at t must equal the sign of next_ret at t
assert (df['target'] == (df['next_ret'] > 0).astype(int)).all()
# next_ret at t must equal ret_1 at t+1
assert ((1 + df['next_ret']).shift(1) - (1 + df['ret_1'])).abs().dropna().max() < 1e-9
assert 'target' not in features and 'next_ret' not in features
print('No forward info in feature list; shift directions verified.')

## Feature groups and correlation

In [ ]:
for name, cols in feature_groups().items():
    print(f'{name:12s}: {cols}')

subset = features[:20]
plt.figure(figsize=(11, 9))
sns.heatmap(df[subset].corr(), cmap='coolwarm', center=0, annot=False)
plt.title('Feature correlations')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_correlation.png', dpi=150)
plt.show()

## Chronological split

In [ ]:
train, test = split_time_series(df, train_fraction=0.8)
print('train:', train.index.min().date(), '->', train.index.max().date(), len(train))
print('test: ', test.index.min().date(), '->', test.index.max().date(), len(test))
print('up rate train/test:', round(train.target.mean(), 4), round(test.target.mean(), 4))